# Machine Learning II — Heart murmur detection from phonocardiograms

**CirCor DigiScope dataset · murmur task of the George B. Moody PhysioNet Challenge 2022 · DCC/FCUP 2026/2027**

The goal is to predict, **for each patient**, whether a heart murmur is *Present*, *Absent* or *Unknown*, from the heart sound recordings (phonocardiograms, PCG) taken at up to four auscultation locations. We implement two deep learning classifiers:

* a **2D convolutional neural network (CNN)** applied to log-mel spectrograms;
* a **recurrent neural network (RNN)**, a bidirectional GRU applied to sequences of MFCC vectors.

Both are evaluated with the challenge metric, the **weighted accuracy** (one prediction per patient).

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Data exploration |
| 2 | Evaluation protocol and metric |
| 3 | Pre-processing and features |
| 4 | CNN: architecture, training, results |
| 5 | RNN: architecture, training, results |
| 6 | Comparison and discussion |
| 7 | Final test evaluation and hidden-set predictions (locked until the final model is chosen) |

**How to run:** see `README.md`. The reusable code (data loading, splits, features, models, training, metric) lives in `src/`; this notebook calls it and shows and justifies every decision. With `RETRAIN = False` the saved models and results are loaded (the whole notebook runs in a few minutes); with `RETRAIN = True` all models are trained again (about 2–3 hours on a laptop CPU).

## 0 — Setup

In [ ]:
import wave

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from IPython.display import display
from scipy.signal import welch

# Run on the CPU, as recommended in the practical classes: on Apple Silicon the tensorflow-metal
# plug-in is known to give wrong results. The dataset is small enough for a CPU.
FORCE_CPU = True
if FORCE_CPU:
    tf.config.set_visible_devices([], "GPU")

from src import config
from src.data import load_dataset, load_segmentation
from src.splits import load_splits, make_splits, test_patients
from src.metrics import confusion_matrix, plot_confusion_matrix, summary, weighted_accuracy
from src.preprocessing import bandpass, load_wav, make_windows, normalize
from src.features import compute_features, logmel, mfcc_sequence
from src.models import build_cnn, build_rnn
from src.training import plot_histories, predict_with_fold_models, run_cv
from src.aggregation import RULES, evaluate_cv, predict_hierarchical, recordings_to_patients, tune_thresholds

RETRAIN = False          # True: train every model again; False: load saved models/results when available
FOLDS_TO_RUN = range(5)  # dev folds used for cross-validation
EVALUATE_TEST = False    # the test set stays locked until the final model is chosen (end of the project)

keras.utils.set_random_seed(config.SEED)
pd.set_option("display.precision", 3)
FIG_DIR = config.RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("Devices:", [d.device_type for d in tf.config.get_visible_devices()])
print("Dataset folder:", config.DATA_DIR)

## 1 — Data exploration

The public part of CirCor DigiScope has **942 patients** and **3163 recordings**. Each patient has a text file (`<id>.txt`) listing their recordings and their metadata and labels; each recording comes as a `.wav` file (4 kHz, 16 bit), a `.hea` header and a `.tsv` segmentation file (start and end of every S1, systole, S2 and diastole).

We use **only the `Murmur` label**. We read the patient files rather than `training_data.csv`, so that exactly the same loader also reads the hidden test set (same layout, no labels).

In [ ]:
def duration(path):
    with wave.open(path) as w:
        return w.getnframes() / w.getframerate()

patients, recordings = load_dataset()
recordings["duration"] = recordings["wav_path"].map(duration)
print(f"{len(patients)} patients, {len(recordings)} recordings")
display(patients.head(3))
display(recordings.head(3))

### 1.1 Class distribution

In [ ]:
counts = patients["murmur"].value_counts().reindex(config.CLASSES)
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(counts.index, counts.values, color=["C3", "C1", "C0"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 10, f"{v} ({v / counts.sum():.1%})", ha="center")
ax.set_ylim(0, counts.max() * 1.15)
ax.set_ylabel("patients")
ax.set_title("Murmur label (one per patient)")
plt.tight_layout(); fig.savefig(FIG_DIR / "class_distribution.png", dpi=150); plt.show()

The classes are strongly **imbalanced**: about 74% of the patients are *Absent*, 19% *Present* and only 7% *Unknown*. Two consequences for the design:

* a network trained without care would mostly predict *Absent* → we use **class weights** in the loss (Sections 4–5);
* the metric (Section 2) counts a *Present* patient 5 times and an *Unknown* patient 3 times, so missing a murmur costs much more than a false alarm → the final decision rule must favour the minority classes, and we **tune its thresholds on validation data**.

### 1.2 Demographics and the other labels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, col in zip(axes, ["age", "sex", "campaign"]):
    pd.crosstab(patients[col].fillna("missing"), patients["murmur"])[config.CLASSES].plot.bar(ax=ax, rot=0)
    ax.set_title(f"{col} by murmur label")
plt.tight_layout(); plt.show()
print("Missing age by pregnancy status:", patients.loc[patients["age"].isna(), "pregnant"].value_counts().to_dict())
display(pd.crosstab(patients["murmur"], patients["outcome"], margins=True))

* Most patients are children; the age is missing almost only for the pregnant patients (70 of 74).
* The clinical `Outcome` (normal/abnormal after a full examination) is **not** the same as the murmur label: 263 *Absent* patients have an abnormal outcome. We do not use it (the project only uses `Murmur`), and it would not be available for a new patient anyway.

### 1.3 Recordings per patient, locations and durations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
recordings.groupby("patient_id").size().value_counts().sort_index().plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Recordings per patient"); axes[0].set_xlabel("number of recordings")
recordings["location"].value_counts().plot.bar(ax=axes[1], rot=0)
axes[1].set_title("Recordings per auscultation location")
for c in config.CLASSES:
    axes[2].hist(recordings.loc[recordings["murmur"] == c, "duration"], bins=40, alpha=0.5, density=True, label=c)
axes[2].axvline(config.WINDOW_SEC, color="k", ls="--", label=f"window = {config.WINDOW_SEC:.0f} s")
axes[2].set_title("Recording duration (s)"); axes[2].legend()
plt.tight_layout(); fig.savefig(FIG_DIR / "recordings_durations.png", dpi=150); plt.show()

print(recordings["duration"].describe().round(1).to_dict())
repeated = recordings.groupby(["patient_id", "location"]).size()
print(f"patients with a repeated location: {(repeated > 1).groupby('patient_id').any().sum()}"
      f" | 'Phc' (other location) recordings: {(recordings['location'] == 'Phc').sum()}")

* Most patients were recorded at the four valves (AV, PV, TV, MV); some have fewer recordings, a few have the same location recorded twice or a recording at another location (`Phc`). Our models therefore work **per recording** and combine any number of recordings per patient (Section 4.3).
* Durations go from 5.2 s to 64.5 s (median about 21 s), so the inputs must be made uniform. We split every recording into **windows of 4 s with 50% overlap**: 4 s contain several cardiac cycles (at least 3 even at 45 bpm, and children's heart rates are higher), the shortest recording still gives one window, and the windows give about 31 000 training examples instead of 3163. At inference time the predictions of all windows are averaged, so the whole recording is used.

### 1.4 Where is the murmur heard?

In [ ]:
present = patients[patients["murmur"] == "Present"]
rec_present = recordings[recordings["murmur"] == "Present"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
present["murmur_locations"].map(len).value_counts().sort_index().plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Present patients: number of locations\nwhere the murmur is heard")
rec_present["murmur_audible_here"].map({True: "murmur heard", False: "murmur not heard"}).value_counts().plot.pie(
    ax=axes[1], autopct="%.0f%%", colors=["C3", "0.7"])
axes[1].set_title("Recordings of Present patients"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()
print(f"{(~rec_present['murmur_audible_here']).sum()} of {len(rec_present)} recordings of Present patients "
      f"were taken at a location where the murmur was NOT heard")

The label belongs to the **patient**, but in 34 of the 179 *Present* patients the murmur is heard at a single location, and about 19% of the recordings of *Present* patients do not contain it. This has two consequences:

* **label noise** when every recording inherits the patient label: some "Present" training windows contain no murmur. The `Murmur locations` field lets us relabel them; this option (`label_mode="refined"`) is implemented and will be tested after the checkpoint;
* **aggregation**: a single recording with a clear murmur is enough to call the patient *Present*. Averaging over the four locations would dilute it, so we take the **maximum** over recordings (Section 4.3).

### 1.5 The same subject in both campaigns

In [ ]:
pairs = patients.dropna(subset=["additional_id"])
label = patients.set_index("patient_id")["murmur"]
other_label = label.loc[pairs["additional_id"].astype(int)].to_numpy()
print(f"{len(pairs)} patients have an 'Additional ID' that is another patient of the public set "
      f"-> {len(pairs) // 2} subjects recorded in both campaigns (CC2014 and CC2015)")
print(f"same murmur label in both campaigns: {(pairs['murmur'].to_numpy() == other_label).sum()} / {len(pairs)}")
display(pd.crosstab(pairs["murmur"].to_numpy(), other_label, rownames=["label"], colnames=["label of the other ID"]))

140 patient IDs correspond to only 70 people, recorded in both data-collection campaigns. If one ID of a person went to the training set and the other to the test set, the model would be tested on a person it has already heard, and the results would be optimistic. Our split is therefore done by **subject**, not by patient ID: the two IDs of a person always go to the same subset (Section 2).

### 1.6 What do the signals look like?

In [ ]:
examples = {
    "Absent": patients[patients["murmur"] == "Absent"].iloc[0],
    "Present": patients[(patients["murmur"] == "Present") & (patients["systolic_grading"] == "III/VI")].iloc[0],
    "Unknown": patients[patients["murmur"] == "Unknown"].iloc[0],
}
STATES = {1: ("S1", "C3"), 2: ("systole", "C1"), 3: ("S2", "C0"), 4: ("diastole", "0.8")}
T0, T1 = 2.0, 6.0     # 4 s shown, the length of one window

fig, axes = plt.subplots(3, 2, figsize=(15, 8.5), gridspec_kw={"width_ratios": [1.5, 1]})
for row, (name, p) in zip(axes, examples.items()):
    recs = recordings[recordings["patient_id"] == p["patient_id"]]
    loc = p["most_audible_location"] if isinstance(p["most_audible_location"], str) else "MV"
    rec = recs[recs["location"] == loc].iloc[0] if (recs["location"] == loc).any() else recs.iloc[0]
    x = normalize(bandpass(load_wav(rec["wav_path"])[0]))[int(T0 * config.SR):int(T1 * config.SR)]
    t = T0 + np.arange(len(x)) / config.SR
    row[0].plot(t, x, lw=0.5, color="k")
    for s in load_segmentation(rec["tsv_path"]).itertuples():
        if s.state in STATES and s.end > T0 and s.start < T1:
            row[0].axvspan(max(s.start, T0), min(s.end, T1), color=STATES[s.state][1], alpha=0.3, lw=0)
    extra = f" — {p['systolic_timing']} murmur, grade {p['systolic_grading']}" if name == "Present" else ""
    row[0].set_title(f"{name}: patient {p['patient_id']}, location {rec['location']}{extra}")
    row[0].set_xlim(T0, T1)
    S = logmel(x[None, :])[0]
    row[1].imshow(S, origin="lower", aspect="auto", extent=[T0, T1, 0, S.shape[0]], cmap="magma")
    row[1].set_title("log-mel spectrogram (32 bands, 25–800 Hz)"); row[1].set_ylabel("mel band")
axes[0, 0].legend([plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.4) for _, c in STATES.values()],
                  [n for n, _ in STATES.values()], loc="upper right", ncol=4, fontsize=8)
axes[-1, 0].set_xlabel("time (s)"); axes[-1, 1].set_xlabel("time (s)")
plt.tight_layout(); fig.savefig(FIG_DIR / "examples.png", dpi=150); plt.show()

Each heartbeat shows two bursts, **S1** (closing of the mitral and tricuspid valves) and **S2** (aortic and pulmonary valves). A systolic murmur is extra sound **between S1 and S2**: in the *Present* example (a holosystolic murmur) the systole is filled with energy, visible in the spectrogram as a band connecting S1 and S2, while in the *Absent* example the systole is nearly silent. Murmurs are therefore **local time-frequency patterns that repeat with every beat**, which motivates a CNN on spectrograms (local patterns, anywhere in the window) and an RNN on frame sequences (the S1–systole–S2–diastole order). *Unknown* recordings are often noisy or of poor quality, which is why the expert could not decide.

### 1.7 Frequency content

In [ ]:
def mean_psd(recs):
    psd = []
    for path in recs["wav_path"]:
        f, p = welch(normalize(load_wav(path)[0]), fs=config.SR, nperseg=1024)
        psd.append(p)
    return f, 10 * np.log10(np.mean(psd, axis=0))

groups = {"Absent": recordings[recordings["murmur"] == "Absent"],
          "Present (murmur heard at this location)": recordings[recordings["murmur_audible_here"]],
          "Unknown": recordings[recordings["murmur"] == "Unknown"]}
fig = plt.figure(figsize=(9, 3.6))
for (name, recs), c in zip(groups.items(), ["C0", "C3", "C1"]):
    f, p = mean_psd(recs)
    plt.plot(f, p, color=c, label=f"{name} ({len(recs)} rec.)")
plt.axvspan(*config.BANDPASS, color="0.9", zorder=0, label="band-pass 25–400 Hz")
plt.xlim(0, 1000); plt.xlabel("frequency (Hz)"); plt.ylabel("mean PSD (dB)")
plt.title("Average power spectral density of the raw recordings"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(FIG_DIR / "psd.png", dpi=150); plt.show()

Most of the energy of heart sounds lies below about 200 Hz, and recordings with a murmur have more energy than *Absent* ones between roughly 100 and 400 Hz (the murmur), while above 400–500 Hz the spectrum is mostly noise. We therefore **band-pass filter between 25 and 400 Hz**: this removes very low frequencies (stethoscope movement, breathing) and high-frequency noise while keeping heart sounds and murmurs. The cut-off frequencies are a hyperparameter that we will revisit (e.g. 25–800 Hz).

## 2 — Evaluation protocol and metric

### 2.1 Data splitting

* **By subject, not by recording:** all the recordings of a patient go to the same subset, and the two patient IDs of the 70 subjects recorded twice (Section 1.5) are merged into one *subject group*.
* **Hold-out test set (20% of the subjects):** never used for any decision (architecture, hyperparameters, pre-processing, early stopping, thresholds); it is evaluated **once**, at the end of the project (Section 7).
* **5-fold cross-validation on the remaining 80% (dev set):** in iteration *k*, fold *k* is the validation set (early stopping, threshold tuning, comparison of models) and the 4 other folds are the training set. We report the mean ± standard deviation of the weighted accuracy over the 5 folds. With only 68 *Unknown* patients, a single validation set (~14 Unknown patients) would give very noisy estimates; 5 folds use every dev patient for validation once.
* **Stratified** by murmur label (`StratifiedGroupKFold`), so every subset has the same class proportions.

The split is computed once with a fixed seed (`src/splits.py`) and saved in `splits/splits.csv`, so that every member of the group uses exactly the same subsets.

In [ ]:
splits = load_splits()
cols = ["patient_id", "subset", "fold"]
fresh = make_splits(patients).sort_values("patient_id")[cols].to_numpy()
assert (splits.sort_values("patient_id")[cols].to_numpy() == fresh).all(), "splits.csv does not match make_splits()"
assert (splits.groupby("subject_group")[["subset", "fold"]].nunique() == 1).all().all(), "a subject is in two subsets"

part = np.where(splits["subset"] == "test", "test", "dev fold " + splits["fold"].astype(str))
table = pd.crosstab(part, splits["murmur"])[config.CLASSES]
table["total"] = table.sum(axis=1)
for c in config.CLASSES:
    table[f"% {c}"] = 100 * table[c] / table["total"]
display(table)
print(f"{splits['subject_group'].nunique()} subject groups; every subject belongs to exactly one subset")

### 2.2 Weighted accuracy

With $m_{ij}$ the number of patients **predicted** as class $i$ whose **true** class is $j$ ($i, j \in \{P, U, A\}$):

$$\mathrm{WA}_\text{murmur} = \frac{5\,m_{PP} + 3\,m_{UU} + m_{AA}}{5\,(m_{PP}+m_{UP}+m_{AP}) + 3\,(m_{PU}+m_{UU}+m_{AU}) + (m_{PA}+m_{UA}+m_{AA})}$$

i.e. the accuracy when each *Present* patient counts 5 times and each *Unknown* patient 3 times. `src/metrics.py` reimplements `compute_weighted_accuracy` of the official evaluation code; `tests/test_metrics.py` checks it against hand-computed cases and against the formula above. To see what a "good" value is, we evaluate trivial classifiers on the dev set:

In [ ]:
dev = patients[patients["patient_id"].isin(splits.loc[splits["subset"] == "dev", "patient_id"])]
rows = [{"classifier": f"always {c}", "WA": weighted_accuracy(dev["murmur"], [c] * len(dev)),
         "accuracy": (dev["murmur"] == c).mean()} for c in config.CLASSES]
priors = dev["murmur"].value_counts(normalize=True).reindex(config.CLASSES).to_numpy()
random_pred = np.random.default_rng(config.SEED).choice(config.CLASSES, size=len(dev), p=priors)
rows.append({"classifier": "random, class proportions", "WA": weighted_accuracy(dev["murmur"], random_pred),
             "accuracy": (dev["murmur"].to_numpy() == random_pred).mean()})
trivial = pd.DataFrame(rows).set_index("classifier")
display(trivial)

Predicting *Absent* for everyone gives 74% accuracy but a weighted accuracy of only 0.39, while predicting **Present for everyone already gives WA ≈ 0.50**: this is the bar a useful model must clearly beat. For reference, the best teams of the 2022 challenge obtained a murmur WA of about 0.78 on the challenge's hidden test set.

## 3 — Pre-processing and features

The same signal pre-processing is used for both networks; only the final representation changes, because each architecture needs a different input format.

1. **No resampling:** all recordings are sampled at 4 kHz; the Nyquist frequency (2 kHz) is far above the band of interest.
2. **Band-pass filter 25–400 Hz** (4th-order Butterworth, applied forward and backward so that S1/S2 are not shifted in time), justified by the spectra of Section 1.7.
3. **Z-score normalisation per recording:** removes gain differences between stethoscopes and recording sessions.
4. **Windows of 4 s with 2 s hop** (Section 1.3); every window inherits the label of its patient.
5. **Features**
   * **CNN — log-mel spectrogram** (FFT of 256 samples = 64 ms, hop 16 ms, 32 mel bands between 25 and 800 Hz, in dB) → a 32 × 251 "image" per window. The murmur appears as a local time-frequency pattern between S1 and S2 (Section 1.6), which is what 2D convolutions detect.
   * **RNN — MFCC sequence** (13 MFCCs + their first and second time derivatives, hop 32 ms) → a sequence of 126 vectors of 39 values per window. MFCCs summarise the spectral envelope of each frame in a few values, and the larger hop halves the number of time steps, which keeps back-propagation through time fast and stable.
6. **Standardisation per feature** (per mel band / per MFCC coefficient) with mean and standard deviation computed **on the training folds only**, then applied to the validation fold (done inside `run_cv`).

Features are cached in `cache/` (not versioned); the cache file name is a hash of every parameter, so changing the pre-processing never reuses stale features.

In [ ]:
rec = recordings.iloc[0]
x_raw, _ = load_wav(rec["wav_path"])
x = normalize(bandpass(x_raw))
W = make_windows(x)
S, M = logmel(W[:1])[0], mfcc_sequence(W[:1])[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
t = np.arange(2 * config.SR) / config.SR
axes[0, 0].plot(t, x_raw[:2 * config.SR], lw=0.5); axes[0, 0].set_title(f"raw signal ({rec['recording_id']}, first 2 s)")
axes[0, 1].plot(t, x[:2 * config.SR], lw=0.5); axes[0, 1].set_title("band-pass 25–400 Hz + z-score")
axes[1, 0].imshow(S, origin="lower", aspect="auto", cmap="magma")
axes[1, 0].set_title(f"CNN input: log-mel of the first window, shape {S.shape}")
axes[1, 0].set_xlabel("frame (16 ms)"); axes[1, 0].set_ylabel("mel band")
axes[1, 1].imshow(((M - M.mean(0)) / M.std(0)).T, origin="lower", aspect="auto", cmap="coolwarm")
axes[1, 1].set_title(f"RNN input: MFCC + Δ + ΔΔ sequence, shape {M.shape} (transposed)")
axes[1, 1].set_xlabel("time step (32 ms)"); axes[1, 1].set_ylabel("feature")
plt.tight_layout(); plt.show()
print(f"{rec['duration']:.1f} s recording -> {len(W)} windows of {config.WINDOW_SEC:.0f} s")

In [ ]:
feats_cnn = compute_features(recordings, "logmel")
feats_rnn = compute_features(recordings, "mfcc")
recordings["windows"] = recordings["recording_id"].map(lambda r: len(feats_cnn[r]))
print(f"{recordings['windows'].sum()} windows | CNN input {feats_cnn[rec['recording_id']].shape[1:]}"
      f" | RNN input {feats_rnn[rec['recording_id']].shape[1:]}")
display(recordings.groupby("murmur")["windows"].agg(["sum", "mean"]).reindex(config.CLASSES)
        .rename(columns={"sum": "windows", "mean": "windows per recording"}))

## 4 — CNN

### 4.1 Architecture

A 2D CNN on the log-mel spectrogram of each window (input 32 × 251 × 1):

* **4 convolutional blocks**: `Conv2D 3×3` → `BatchNormalization` → `ReLU` → `MaxPooling 2×2`, with 16, 32, 64 and 128 feature maps. Small 3×3 filters stacked in depth (as in VGG) grow the receptive field while keeping few parameters; pooling halves both axes at each block (32 × 251 → 2 × 15) and gives some invariance to small shifts in time and frequency.
* **Global average pooling** instead of `Flatten` + `Dense`: one value per feature map, i.e. "is this pattern present somewhere in the window", wherever the murmur occurs. It also keeps the network small (~98 000 parameters), which matters with only ~600 training patients per fold.
* `Dropout(0.3)` and a `Dense(3, softmax)` output (one probability per class).

In [ ]:
build_cnn().summary()

### 4.2 Training strategy

| Choice | Value | Why |
|---|---|---|
| Loss | categorical cross-entropy with **class weights** `balanced` | each class contributes the same total weight despite the imbalance (weights ≈ 1.8 / 6.8 / 0.44 for Present / Unknown / Absent windows) |
| Optimizer | Adam, learning rate 10⁻³ | adaptive learning rate; it was the most robust optimizer in the practical classes (P03) |
| Mini-batch | 64 windows | about 300 updates per epoch |
| Epochs | at most 40, **early stopping on the patient-level WA** of the validation fold (patience 8), best weights restored | after each epoch we aggregate the validation predictions per patient and compute the challenge metric; we stop on what we are evaluated on, not on the window-level loss |
| Regularisation | batch normalisation, dropout 0.3, L2 (10⁻⁴) on the convolution kernels, global average pooling, early stopping | small dataset → strong risk of overfitting |
| Data augmentation | none yet | planned (random crops, gain, noise, SpecAugment masks) |

These values are a reasonable first configuration, **not tuned yet**; they will be tuned using the cross-validation score only.

In [ ]:
oof_cnn, hist_cnn = run_cv("cnn", build_cnn, feats_cnn, recordings, splits, folds=FOLDS_TO_RUN,
                           std_axis=(0, 2), add_channel=True, retrain=RETRAIN)
fig = plot_histories(hist_cnn, "CNN — training curves of the 5 folds")
fig.savefig(FIG_DIR / "cnn_curves.png", dpi=150); plt.show()
display(pd.DataFrame({k: {"best epoch": h["best_epoch"], "epochs run": len(h["loss"]),
                          "best val patient WA (thresholds 0.5)": h["best_val_patient_wa"],
                          "minutes": h["seconds"] / 60} for k, h in hist_cnn.items()}).T)

*(interpretation of the results: cnn_curves)*

### 4.3 From windows to one prediction per patient

1. **window → recording:** average of the softmax probabilities of the recording's windows;
2. **recording → patient**, hierarchical rule (as in the challenge's baseline approaches):
   *Present* if $\max_\text{rec} P(\text{Present}) > \tau_P$; otherwise *Unknown* if $\max_\text{rec} P(\text{Unknown}) > \tau_U$; otherwise *Absent*.
   The **max** over recordings follows from Section 1.4: one location with a murmur is enough.

We compare three variants:

* `argmax_mean`: class with the highest probability averaged over all recordings (simple baseline);
* `hierarchical_fixed`: hierarchical rule with $\tau_P = \tau_U = 0.5$;
* `hierarchical_tuned`: thresholds chosen by grid search to maximise the WA. To avoid an optimistic estimate, the thresholds used on fold *k* are tuned on the validation predictions of the **other** folds (cross-fitting); the final thresholds for the test set will be tuned on all validation folds.

In [ ]:
def cv_report(name, oof):
    # Patient-level evaluation of the cross-validation predictions of one model.
    pats, per_fold = evaluate_cv(oof)
    display(per_fold.set_index("fold"))
    wa = per_fold[[f"WA_{r}" for r in RULES]]
    print("weighted accuracy over the folds (mean ± std):")
    for r in RULES:
        print(f"  {r:<20} {wa[f'WA_{r}'].mean():.3f} ± {wa[f'WA_{r}'].std():.3f}")
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for ax, r in zip(axes, RULES):
        A = confusion_matrix(pats["murmur"], pats[f"pred_{r}"])
        plot_confusion_matrix(A, title=f"{r}\n(5 val. folds pooled)", ax=ax)
    fig.suptitle(f"{name.upper()} — patient-level confusion matrices (rows: predicted, columns: true)")
    plt.tight_layout(); fig.savefig(FIG_DIR / f"{name}_confusion.png", dpi=150); plt.show()
    return pats, per_fold

pats_cnn, folds_cnn = cv_report("cnn", oof_cnn)

*(interpretation of the results: cnn_results)*

### 4.4 What does the network see at the recording level?

The probability of *Present* for each recording, grouped by what we know about it: does the CNN react to the murmur where it is actually heard?

In [ ]:
def plot_recording_probs(name, oof):
    groups = {"Absent patient": oof["murmur"] == "Absent",
              "Unknown patient": oof["murmur"] == "Unknown",
              "Present patient,\nmurmur not heard here": (oof["murmur"] == "Present") & ~oof["recording_id"].map(
                  recordings.set_index("recording_id")["murmur_audible_here"]),
              "Present patient,\nmurmur heard here": oof["recording_id"].map(
                  recordings.set_index("recording_id")["murmur_audible_here"])}
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.boxplot([oof.loc[m, "p_Present"] for m in groups.values()], showfliers=False)
    ax.set_xticks(range(1, len(groups) + 1), [f"{g}\n(n={m.sum()})" for g, m in groups.items()], fontsize=8)
    ax.set_ylabel("P(Present) of the recording")
    ax.set_title(f"{name.upper()} — recording-level probability of Present (validation folds)")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout(); fig.savefig(FIG_DIR / f"{name}_recording_probs.png", dpi=150); plt.show()

plot_recording_probs("cnn", oof_cnn)

*(interpretation of the results: cnn_recordings)*

## 5 — RNN

### 5.1 Architecture

A recurrent network on the MFCC sequence of each window (126 time steps × 39 features):

* **GRU** units rather than a vanilla RNN: the gates let the network keep information over the 126 steps without vanishing gradients, with fewer parameters than an LSTM;
* **bidirectional**: the whole window is available, and what comes *after* a sound (e.g. S2 after a systolic murmur) is as informative as what comes before;
* **two stacked layers** of 64 units per direction: the first returns its whole output sequence, the second only its final state, a summary of the window;
* `Dropout(0.3)` and a `Dense(3, softmax)` output. About 115 000 parameters, similar to the CNN, so that the comparison is fair.

In [ ]:
build_rnn().summary()

### 5.2 Training strategy

Same loss (cross-entropy with balanced class weights), optimizer (Adam, 10⁻³), mini-batch (64), early stopping on the validation patient-level WA and aggregation rule as the CNN, so that the two models differ only in the input representation and the architecture. Additionally, **gradient clipping** (norm ≤ 1) protects against exploding gradients, the classical failure mode of recurrent networks trained with back-propagation through time.

In [ ]:
oof_rnn, hist_rnn = run_cv("rnn", build_rnn, feats_rnn, recordings, splits, folds=FOLDS_TO_RUN,
                           std_axis=(0, 1), retrain=RETRAIN, clipnorm=1.0)
fig = plot_histories(hist_rnn, "RNN — training curves of the 5 folds")
fig.savefig(FIG_DIR / "rnn_curves.png", dpi=150); plt.show()
display(pd.DataFrame({k: {"best epoch": h["best_epoch"], "epochs run": len(h["loss"]),
                          "best val patient WA (thresholds 0.5)": h["best_val_patient_wa"],
                          "minutes": h["seconds"] / 60} for k, h in hist_rnn.items()}).T)

*(interpretation of the results: rnn_curves)*

### 5.3 Patient-level results

In [ ]:
pats_rnn, folds_rnn = cv_report("rnn", oof_rnn)

In [ ]:
plot_recording_probs("rnn", oof_rnn)

*(interpretation of the results: rnn_results)*

## 6 — Comparison and discussion

Cross-validation results on the dev set (5 validation folds). The main metric is the weighted accuracy; the other columns are **additional** metrics, computed on the pooled predictions of the 5 folds with the `hierarchical_tuned` rule.

In [ ]:
rows = []
for name, pats, per_fold, hist in [("CNN", pats_cnn, folds_cnn, hist_cnn), ("RNN", pats_rnn, folds_rnn, hist_rnn)]:
    row = {"model": name, "parameters": hist[0]["params"],
           "minutes per fold": np.mean([h["seconds"] for h in hist.values()]) / 60}
    for r in RULES:
        row[f"WA {r}"] = f"{per_fold[f'WA_{r}'].mean():.3f} ± {per_fold[f'WA_{r}'].std():.3f}"
    s = summary(pats["murmur"], pats["pred_hierarchical_tuned"])
    row.update({k: s[k] for k in ["accuracy", "recall_Present", "recall_Unknown", "recall_Absent", "macro_f1"]})
    rows.append(row)
comparison = pd.DataFrame(rows).set_index("model")
comparison.to_csv(config.RESULTS_DIR / "cv_summary.csv")
display(comparison)
print(f"reference: always Present -> WA {trivial.loc['always Present', 'WA']:.3f} on the dev set")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (name, pats) in zip(axes, [("CNN", pats_cnn), ("RNN", pats_rnn)]):
    plot_confusion_matrix(confusion_matrix(pats["murmur"], pats["pred_hierarchical_tuned"]),
                          title=f"{name} (hierarchical_tuned)", ax=ax)
plt.tight_layout(); fig.savefig(FIG_DIR / "comparison_confusion.png", dpi=150); plt.show()

*(interpretation of the results: discussion)*

## 7 — Final test evaluation and hidden-set predictions

**Locked until the end of the project.** Once all the decisions have been taken with cross-validation, each model is evaluated **once** on the hold-out test set:

* final model = **ensemble of the 5 fold models** (their window probabilities are averaged), each using the standardisation statistics of its own training folds;
* thresholds of the hierarchical rule tuned on the validation predictions of all 5 folds (never on the test set).

The same procedure produces the `.csv` file for the hidden set of the bonus competition (one row per patient: `patient_id,murmur`).

In [ ]:
FINAL = {"cnn": dict(feats=feats_cnn, oof=oof_cnn, std_axis=(0, 2), add_channel=True),
         "rnn": dict(feats=feats_rnn, oof=oof_rnn, std_axis=(0, 1), add_channel=False)}

def final_predictions(name, target_recordings, feats):
    cfg = FINAL[name]
    tau_p, tau_u, _ = tune_thresholds(recordings_to_patients(cfg["oof"]))   # validation folds only
    rec_probs = predict_with_fold_models(name, feats, recordings, splits, target_recordings,
                                         std_axis=cfg["std_axis"], add_channel=cfg["add_channel"])
    pats = recordings_to_patients(rec_probs)
    pats["pred"] = predict_hierarchical(pats, tau_p, tau_u)
    return pats

if EVALUATE_TEST:
    test_recs = recordings[recordings["patient_id"].isin(test_patients(splits))]
    fig, axes = plt.subplots(1, len(FINAL), figsize=(4.5 * len(FINAL), 3.8))
    for ax, name in zip(axes, FINAL):
        pats = final_predictions(name, test_recs, FINAL[name]["feats"])
        print(name.upper(), {k: round(v, 3) for k, v in summary(pats["murmur"], pats["pred"]).items()})
        plot_confusion_matrix(confusion_matrix(pats["murmur"], pats["pred"]), title=f"{name.upper()} — TEST", ax=ax)
    plt.tight_layout(); fig.savefig(FIG_DIR / "test_confusion.png", dpi=150); plt.show()
else:
    print("Test set locked (EVALUATE_TEST = False).")

In [ ]:
HIDDEN_DIR = None        # folder of the hidden set (shared later in the semester)
BEST_MODEL = "cnn"       # to be chosen with the cross-validation results

if HIDDEN_DIR is not None:
    _, hidden_recs = load_dataset(HIDDEN_DIR)
    kind = "logmel" if BEST_MODEL == "cnn" else "mfcc"
    feats = {**FINAL[BEST_MODEL]["feats"], **compute_features(hidden_recs, kind)}
    pats = final_predictions(BEST_MODEL, hidden_recs, feats)
    out = pats[["patient_id"]].assign(murmur=pats["pred"])
    out.to_csv(config.RESULTS_DIR / "hidden_predictions.csv", index=False)
    display(out["murmur"].value_counts())
else:
    print("No hidden set yet (HIDDEN_DIR = None).")